In [ ]:
!pip install pandas

In [76]:
import json
import psycopg2
from pathlib import Path


DB_CONFIG = {
    "host": "localhost",
    "port": 55432,
    "database": "supply_chain",
    "user": "supplychain_app",
    "password": "Cr7@1034"
}


OUTPUT_FILE = Path(
    "output/database_schema.json"
)



def get_connection():

    return psycopg2.connect(
        **DB_CONFIG
    )



def get_tables(cursor):

    cursor.execute(
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema='public'
        AND table_type='BASE TABLE'
        ORDER BY table_name;
        """
    )

    return [
        row[0]
        for row in cursor.fetchall()
    ]



def get_columns(cursor, table_name):

    cursor.execute(
        """
        SELECT

            column_name,

            data_type,

            character_maximum_length,

            is_nullable,

            column_default

        FROM information_schema.columns

        WHERE table_schema='public'

        AND table_name=%s

        ORDER BY ordinal_position;

        """,
        (
            table_name,
        )
    )


    return cursor.fetchall()



def get_primary_keys(cursor, table_name):

    cursor.execute(
        """
        SELECT
            kcu.column_name

        FROM
            information_schema.table_constraints tc

        JOIN
            information_schema.key_column_usage kcu

        ON
            tc.constraint_name=kcu.constraint_name

        WHERE
            tc.table_name=%s

        AND
            tc.constraint_type='PRIMARY KEY';

        """,
        (
            table_name,
        )
    )


    return [
        row[0]
        for row in cursor.fetchall()
    ]



def get_foreign_keys(cursor, table_name):

    cursor.execute(
        """
        SELECT

            kcu.column_name,

            ccu.table_name AS referenced_table,

            ccu.column_name AS referenced_column


        FROM
            information_schema.table_constraints tc


        JOIN
            information_schema.key_column_usage kcu

        ON
            tc.constraint_name=kcu.constraint_name


        JOIN
            information_schema.constraint_column_usage ccu

        ON
            ccu.constraint_name=tc.constraint_name


        WHERE
            tc.constraint_type='FOREIGN KEY'

        AND
            tc.table_name=%s;

        """,
        (
            table_name,
        )
    )


    return cursor.fetchall()



def build_create_table(cursor, table):


    columns=get_columns(
        cursor,
        table
    )


    primary_keys=get_primary_keys(
        cursor,
        table
    )


    foreign_keys=get_foreign_keys(
        cursor,
        table
    )



    ddl=[]


    ddl.append(
        f"CREATE TABLE {table} ("
    )



    column_lines=[]


    for column in columns:


        name=column[0]

        dtype=column[1]

        length=column[2]

        nullable=column[3]

        default=column[4]



        if length:

            dtype=f"{dtype}({length})"



        line=f"    {name} {dtype}"



        if nullable=="NO":

            line+=" NOT NULL"



        if default:

            line+=f" DEFAULT {default}"



        column_lines.append(
            line
        )



    if primary_keys:

        column_lines.append(

            "    PRIMARY KEY ("
            +
            ", ".join(primary_keys)
            +
            ")"

        )



    for fk in foreign_keys:


        column_lines.append(

            f"    FOREIGN KEY ({fk[0]}) "
            f"REFERENCES {fk[1]}({fk[2]})"

        )



    ddl.append(
        ",\n".join(column_lines)
    )


    ddl.append(
        ");"
    )


    return "\n".join(ddl)




def main():

    conn=get_connection()

    cursor=conn.cursor()


    try:


        tables=get_tables(
            cursor
        )


        schema_json={}



        for table in tables:


            print(
                "Exporting:",
                table
            )


            schema_json[table]=build_create_table(
                cursor,
                table
            )



        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:


            json.dump(
                schema_json,
                f,
                indent=4
            )



        print(
            "\nSchema exported:",
            OUTPUT_FILE
        )



    finally:

        cursor.close()

        conn.close()



if __name__=="__main__":

    main()

Exporting: customers
Exporting: drivers
Exporting: event_outbox
Exporting: inventory
Exporting: inventory_adjustments
Exporting: inventory_allocations
Exporting: inventory_locations
Exporting: inventory_reservations
Exporting: inventory_snapshots
Exporting: inventory_transactions
Exporting: order_items
Exporting: orders
Exporting: package_items
Exporting: packages
Exporting: payments
Exporting: products
Exporting: purchase_order_items
Exporting: purchase_orders
Exporting: shipment_checkpoints
Exporting: shipment_items
Exporting: shipment_loading_events
Exporting: shipment_tracking
Exporting: shipment_transportation
Exporting: shipments
Exporting: suppliers
Exporting: trailers
Exporting: vehicles
Exporting: warehouse_locations
Exporting: warehouse_tasks
Exporting: warehouses
Exporting: worker_attendance
Exporting: worker_productivity
Exporting: workers

Schema exported: output\database_schema.json


In [2]:
import os

In [8]:
import psycopg2

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": int(os.getenv("DB_PORT", 55432)),
    "database": os.getenv("DB_NAME", "supply_chain"),
    "user": os.getenv("DB_USER", "supplychain_app"),
    "password": os.getenv("DB_PASSWORD", "Cr7@1034"),
}

def execute_sql_file(file_path: str):
    conn = None
    cursor = None
    try:
        # Read the SQL file
        with open(file_path, "r", encoding="utf-8") as file:
            sql_script = file.read()

        # Connect to PostgreSQL
        conn = psycopg2.connect(**DB_CONFIG)
        cursor = conn.cursor()

        # Execute the SQL transaction
        cursor.execute(sql_script)
        conn.commit()
        print("Data inserted successfully.")

    except Exception as error:
        if conn:
            conn.rollback()
        print(f"Failed to insert data: {error}")

    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

if __name__ == "__main__":
    # Ensure your insert statements are saved in 'inserts.sql'
    execute_sql_file("C:\\Users\\rohit\\Downloads\\insert_customers.sql")

Data inserted successfully.


In [77]:
import random
from datetime import datetime, timezone

from core.db import Database


ZONES = [
    "A",
    "B",
    "C",
    "D"
]


STORAGE_TYPES = [
    "RACK",
    "SHELF",
    "BIN",
    "COLD_STORAGE"
]


def generate_locations():

    with Database() as db:


        warehouses = db.fetch_all(
            """
            SELECT warehouse_id
            FROM warehouses
            """
        )


        total = 0


        for warehouse in warehouses:


            warehouse_id = warehouse["warehouse_id"]


            for zone in ZONES:


                for aisle in range(1,6):


                    for rack in range(1,6):


                        for shelf in range(1,3):


                            location_id = (

                                f"LOC-{warehouse_id}-"
                                f"{zone}-"
                                f"{aisle:02d}-"
                                f"{rack:02d}-"
                                f"{shelf:02d}"

                            )


                            db.execute(

                                """
                                INSERT INTO warehouse_locations
                                (
                                    location_id,
                                    warehouse_id,
                                    zone,
                                    aisle,
                                    rack,
                                    shelf,
                                    bin,
                                    storage_type,
                                    capacity_units,
                                    current_utilization,
                                    status
                                )

                                VALUES
                                (
                                    %s,%s,%s,%s,%s,%s,%s,
                                    %s,%s,%s,%s
                                )

                                ON CONFLICT(location_id)
                                DO NOTHING

                                """,

                                (

                                    location_id,

                                    warehouse_id,

                                    zone,

                                    str(aisle),

                                    str(rack),

                                    str(shelf),

                                    f"B-{random.randint(1,20)}",

                                    random.choice(
                                        STORAGE_TYPES
                                    ),

                                    random.randint(
                                        100,
                                        1000
                                    ),

                                    0,

                                    "ACTIVE"

                                )

                            )


                            total += 1



        print(
            f"Created locations : {total}"
        )



if __name__ == "__main__":

    generate_locations()

Created locations : 5000


In [7]:
def bulk_insert_csv(csv_filepath, table_name):
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()

    try:
        with open(csv_filepath, "r", encoding="utf-8") as f:
            # Assumes CSV header column names match database column names
            sql = f"COPY {table_name} FROM STDIN WITH (FORMAT csv, HEADER true)"
            cursor.copy_expert(sql=sql, file=f)

        conn.commit()
        print(f"Successfully inserted {csv_filepath} into {table_name}.")

    except Exception as e:
        conn.rollback()
        print(f"Error inserting data: {e}")

    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    bulk_insert_csv("C:\\Users\\rohit\\Downloads\\customers.csv", "customers")

Error inserting data: missing data for column "created_at"
CONTEXT:  COPY customers, line 2: "CUST-0000001,Samuel,Martin,samuel.martin@aol.com,(274) 748-2542,USA,Florida,Miami,Consumer"



In [28]:
from core.db import Database

# ---------------------------------------------
# MASTER TABLES - NEVER DELETE
# ---------------------------------------------
MASTER_TABLES = {
    "customers",
    "drivers",
    "products",
    "suppliers",
    "trailers",
    "vehicles",
    "warehouse_locations",
    "warehouses",
    "workers",
}

# ---------------------------------------------
# ALL TABLES (36 Database Tables)
# ---------------------------------------------
ALL_TABLES = {
    "customers",
    "delivery_confirmation",
    "drivers",
    "event_outbox",
    "inventory",
    "inventory_adjustments",
    "inventory_allocations",
    "inventory_locations",
    "inventory_reservations",
    "inventory_snapshots",
    "inventory_transactions",
    "order_items",
    "orders",
    "outbound_fulfillment",
    "outbound_shipments",
    "package_items",
    "packages",
    "payments",
    "products",
    "purchase_order_items",
    "purchase_orders",
    "shipment_checkpoints",
    "shipment_items",
    "shipment_loading_events",
    "shipment_tracking",
    "shipment_transportation",
    "shipments",
    "suppliers",
    "trailers",
    "vehicles",
    "warehouse_locations",
    "warehouse_tasks",
    "warehouses",
    "worker_attendance",
    "worker_productivity",
    "workers",
}


# ---------------------------------------------
# CLEAR DATA
# ---------------------------------------------
def reset_transaction_tables():
    tables_to_clear = sorted(ALL_TABLES - MASTER_TABLES)

    print("\n------------------------------------")
    print(f"Master Tables Preserved ({len(MASTER_TABLES)}):")
    for table in sorted(MASTER_TABLES):
        print(f"  [KEEP] {table}")

    print(f"\nTransactional Tables to Clear ({len(tables_to_clear)}):")
    for table in tables_to_clear:
        print(f"  [CLEAR] {table}")
    print("------------------------------------\n")

    with Database() as db:
        try:
            # Batching all tables into a single TRUNCATE statement avoids
            # foreign key dependency failures between transactional tables.
            tables_sql = ", ".join(tables_to_clear)

            print("Truncating transactional tables...")
            db.execute(
                f"""
                TRUNCATE TABLE {tables_sql}
                RESTART IDENTITY
                CASCADE;
                """
            )

            print("\n====================================")
            print("Transactional tables cleared successfully.")
            print("Master data preserved.")
            print("====================================\n")

        except Exception as e:
            print(f"Failed to clear transactional tables: {e}")


if __name__ == "__main__":
    reset_transaction_tables()


------------------------------------
Master Tables Preserved (9):
  [KEEP] customers
  [KEEP] drivers
  [KEEP] products
  [KEEP] suppliers
  [KEEP] trailers
  [KEEP] vehicles
  [KEEP] warehouse_locations
  [KEEP] warehouses
  [KEEP] workers

Transactional Tables to Clear (27):
  [CLEAR] delivery_confirmation
  [CLEAR] event_outbox
  [CLEAR] inventory
  [CLEAR] inventory_adjustments
  [CLEAR] inventory_allocations
  [CLEAR] inventory_locations
  [CLEAR] inventory_reservations
  [CLEAR] inventory_snapshots
  [CLEAR] inventory_transactions
  [CLEAR] order_items
  [CLEAR] orders
  [CLEAR] outbound_fulfillment
  [CLEAR] outbound_shipments
  [CLEAR] package_items
  [CLEAR] packages
  [CLEAR] payments
  [CLEAR] purchase_order_items
  [CLEAR] purchase_orders
  [CLEAR] shipment_checkpoints
  [CLEAR] shipment_items
  [CLEAR] shipment_loading_events
  [CLEAR] shipment_tracking
  [CLEAR] shipment_transportation
  [CLEAR] shipments
  [CLEAR] warehouse_tasks
  [CLEAR] worker_attendance
  [CLEAR] wo